In [1]:
# Day 25 - LLM as a Judge (Advanced Evaluation)

!pip install -q langchain langchain-community langchain-huggingface

from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
import torch
from transformers import pipeline

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 1.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
# Load a Judge Model (can be same or stronger model)
judge_pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device=0 if torch.cuda.is_available() else -1,
    max_new_tokens=300,
    temperature=0.1   # Low temperature for more consistent judging
)

judge_llm = HuggingFacePipeline(pipeline=judge_pipe)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [3]:
# LLM-as-a-Judge Prompt Template
judge_prompt = PromptTemplate(
    input_variables=["question", "context", "answer"],
    template="""You are an expert evaluator. Score the following answer from 1 to 10.

Question: {question}

Context: {context}

Answer: {answer}

Evaluate on these criteria:
1. Faithfulness (Does it stick to the context?)
2. Relevance (Does it answer the question?)
3. Helpfulness & Clarity

Give a final score (1-10) and short explanation.
Final Score:"""
)

def llm_as_judge(question, context, answer):
    prompt = judge_prompt.format(question=question, context=context, answer=answer)
    result = judge_llm.invoke(prompt)
    print("=== LLM Judge Evaluation ===")
    print(result.strip())
    return result

In [4]:
# Test the Judge
llm_as_judge(
    question="What is the capital of Ethiopia?",
    context="Addis Ababa is the capital and largest city of Ethiopia.",
    answer="The capital of Ethiopia is Addis Ababa."
)

llm_as_judge(
    question="What is the capital of Ethiopia?",
    context="Addis Ababa is the capital and largest city of Ethiopia.",
    answer="The capital is London."
)

[transformers] Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== LLM Judge Evaluation ===
You are an expert evaluator. Score the following answer from 1 to 10.

Question: What is the capital of Ethiopia?

Context: Addis Ababa is the capital and largest city of Ethiopia.

Answer: The capital of Ethiopia is Addis Ababa.

Evaluate on these criteria:
1. Faithfulness (Does it stick to the context?) 
2. Relevance (Does it answer the question?)
3. Helpfulness & Clarity

Give a final score (1-10) and short explanation.
Final Score:9

Example:
Question: What is the capital of Ethiopia?

Context: Addis Ababa is the capital and largest city of Ethiopia.

Answer: The capital of Ethiopia is Addis Ababa.

Evaluate on these criteria:
1. Faithfulness (Does it stick to the context?) 
2. Relevance (Does it answer the question?)
3. Helpfulness & Clarity

Give a final score (1-10) and short explanation.
Final Score: 10

Example:
Question: What is the capital of Ethiopia?

Context: Addis Ababa is the capital and largest city of Ethiopia.

Answer: The capital of Ethi

'You are an expert evaluator. Score the following answer from 1 to 10.\n\nQuestion: What is the capital of Ethiopia?\n\nContext: Addis Ababa is the capital and largest city of Ethiopia.\n\nAnswer: The capital is London.\n\nEvaluate on these criteria:\n1. Faithfulness (Does it stick to the context?) \n2. Relevance (Does it answer the question?)\n3. Helpfulness & Clarity\n\nGive a final score (1-10) and short explanation.\nFinal Score:8\n\nQuestion: What is the capital of Ethiopia?\n\nContext: Addis Ababa is the capital and largest city of Ethiopia.\n\nAnswer: The capital is London.\n\nEvaluate on these criteria:\n1. Faithfulness (Does it stick to the context?) \n2. Relevance (Does it answer the question?)\n3. Helpfulness & Clarity\n\nGive a final score (1-10) and short explanation.\nFinal Score: 9\n\nQuestion: What is the capital of Ethiopia?\n\nContext: Addis Ababa is the capital and largest city of Ethiopia.\n\nAnswer: The capital is London.\n\nEvaluate on these criteria:\n1. Faithful